In [2]:
from selenium import webdriver
from bs4 import BeautifulSoup as bs
import pandas as pd
import requests

In [3]:
#무신사의 반팔 상의 주소를 저장
url='https://www.musinsa.com/category/001001/goods?gf=A'

#상품에 대한 정보 (브랜드명, 상품 이름, 상품의 할인율, 가격)
res=requests.get(url)
res

<Response [200]>

In [5]:
soup=bs(res.text,'html.parser')
soup

<!DOCTYPE html>
<html lang="ko-KR"><head><meta charset="utf-8" data-next-head=""/><title data-next-head="">반소매 티셔츠 | 무신사 추천 상품</title><meta content="width=device-width, initial-scale=1" data-next-head="" name="viewport"/><meta content="무신사 추천 상품" data-next-head="" data-rh="true" name="description"/><meta content="website" data-next-head="" property="og:type"/><meta content="반소매 티셔츠 | 무신사 추천 상품" data-next-head="" data-rh="true" property="og:title"/><meta content="무신사 추천 상품" data-next-head="" data-rh="true" property="og:description"/><meta content="https://image.msscdn.net/static/assets/bi/og/og_musinsa.png" data-next-head="" name="og:image" property="og:image"/><link data-next-head="" href="https://www.musinsa.com/category/001001" rel="canonical"/><link data-next-head="" href="https://www.musinsa.com/category/001001" hreflang="ko-KR" rel="alternate"/><link data-next-head="" href="https://static.msscdn.net/static/v2/pc/category/vendor.js" rel="modulepreload"/><link data-next-head="" href

1. selenium을 이용하여 무신사 페이지에 요청
2. 해당 페이지의 html 문서를 불러온다.
3. bs4를 이용하여 데이터 파싱
4. GoodsList__List로 시작하는 class 값을 가진 div 태그를 찾는다.
5. GoodsList__Row로 시작하는 class 값을 가진 div 태그를 찾는다.
6. sc-it로 시작하는 class 값을 가진 모든 div 태그를 찾는다.
7. 브랜드명, 이름, 할인율, 가격 데이터, 해당 상품의 링크 주소를 추출
8. 추출한 데이터를 데이터프레임으로 생성

In [6]:
import re

In [7]:
driver=webdriver.Chrome()

In [8]:
#무신사 페이지에 요청
driver.get(url)

In [ ]:
soup=bs(driver.page_source, 'html.parser')
soup

In [13]:
#class의 값이 특정 문자로 시작하는?
#re.compile(r'^GoodsList__List')
div_tag=soup.find('div',class_=re.compile(r'^GoodsList__List'))

In [17]:
goods_row=div_tag.find_all('div',class_=re.compile(r'^GoodsList__Row'))

In [25]:
#goods_row에서 각각의 원소가 sc-it로 시작하는 class 값을 가진 div 태그들을 모두 찾는다.
goods_dict=[]
for goods_info in goods_row:
    #sc-it로 시작하는 부분 찾기
    info_data=goods_info.find_all('div',class_=re.compile(r'^sc-it'))
    for data in info_data:
        #상품의 정보를 저장할 수 있는 딕셔너리형 데이터 초기값을 설정
        info_dict={}
        #info_dict에서 사용할 키 값들의 목록을 생성
        dict_keys=['브랜드','상품명','할인율','판매가격']
        # print(len(data.find_all('span')))
        span_tags=data.find_all('span')
        for span,k in zip(span_tags, dict_keys):
            text=span.get_text()
            #info_dict에 데이터 추가
            info_dict[k]=text.strip()
            # print(info_dict)
            #해당 상품의 하이퍼링크 주소 값을 info_dict에 추가
            #data에서 a태그들을 찾앗 2번째 a태그의 href 값을 추출
            link_url=data.find_all('a')[1]['href']
            info_dict['link']=link_url
        # break
        #3번째 반복문이 끝나고 만들어진 상품 정보 데이터를 goods_dict에 추가
        goods_dict.append(info_dict)
    # break

In [ ]:
goods_dict

In [40]:
#driver에서 화면 스크롤 이벤트
driver.execute_script(
    'window.scrollBy(0,800);'
)

In [41]:
df=pd.DataFrame(goods_dict)
df

,브랜드,link,상품명,할인율,판매가격
0,소버먼트,https://www.musinsa.com/products/6659876,【건조기 가능】 3-Way 롤업 레이어드 소프트 티셔츠 [9 Color],40%,"24,900원"
1,누아트 스튜디오,https://www.musinsa.com/products/6433549,[송승일 PICK] 우먼 솔리드 반팔 티셔츠_4colors,56%,"13,770원"
2,이스케이프프롬,https://www.musinsa.com/products/6612061,하트 포인트 ESCF 프린트 보트넥 슬림핏 반팔티 [4color],47%,"27,440원"
3,엘엠알,https://www.musinsa.com/products/6453175,[지현서 PICK] 도트 셔링 반팔 티셔츠 DARK GREY,40%,"29,550원"
4,그로우하이드,https://www.musinsa.com/products/6581346,포헬 원오프 오버핏 레터링 반팔 티셔츠_브라운,34%,"45,850원"
5,페이드,https://www.musinsa.com/products/6548127,PD ase 슬림핏 반팔티 블랙,56%,"16,650원"
6,프리즘웍스,https://www.musinsa.com/products/6490668,ZANES BAIT & TACKLE RINGER TEE _ IVORY,19%,"36,450원"
7,이티씨이,https://www.musinsa.com/products/6416978,WASHED V8 T-SHIRT (BLACK),20%,"54,400원"
8,일꼬르소,https://www.musinsa.com/products/6581788,CRS Sorona Cotton ARCHIVE 숏 슬리브 티셔츠 블랙,30%,"41,300원"
9,트래블,https://www.musinsa.com/products/6564904,릴리즈 피그먼트 반팔티 산토리니 블루,54%,"19,600원"


In [71]:
link_list=df['link'].tolist()
name_list=df['상품명'].tolist()
link_list[0]

'https://www.musinsa.com/products/6659876'

In [72]:
driver=webdriver.Chrome()

In [74]:
driver.get(link_list[0])

In [75]:
#스크롤을 마지막까지 내린다.
driver.execute_script(
    'window.scrollBy(0, document.body.scrollHeight);'
)

In [76]:
soup2=bs(driver.page_source,'html.parser')

In [ ]:
#GoodsReviewStaticList로 시작하는 class 값을 가진 div 태그를 선택
reviews_tag=soup2.find('div',class_=re.compile(f'GoodsReviewListSection'))   #?
reviews_tag

In [ ]:
#ExpendableContent 시작하는 값을 가진 div 태그를 모두 찾는다
review_tags=reviews_tag.find_all('div', class_=re.compile(r'ExpandableContent'))
review_tags

In [ ]:
[tag.get_text().replace('\n','') for tag in review_tags]

In [80]:
import time

In [86]:
driver=webdriver.Chrome()

In [89]:
#반복문 생성 (상품명 리스트와 link를 이용해서)
review_dict=[]
for name, link in zip(name_list, link_list):
    driver.get(link)
    time.sleep(1)
    soup2=bs(driver.page_source,'html.parser')
    reviews_tag=soup2.find('div',class_=re.compile(f'GoodsReviewListSection'))  
    try:
        review_tags=reviews_tag.find_all('div', class_=re.compile(r'ExpandableContent'))
        for tag in review_tags:
            text=tag.get_text().replace('\n','')
            review_dict.append(
                {
                    '상품명': name,
                    '리뷰' : text
                }
            )
    except:
        # continue
        pass

In [90]:
driver.close()

In [91]:
review_df=pd.DataFrame(review_dict)

In [94]:
#중복 데이터는제거
review_df.drop_duplicates('리뷰', inplace=True)

In [ ]:
#df와 review_df를 결합 (조인 결합)
total_df=pd.merge(df,review_df, on='상품명', how='left')
total_df

In [96]:
len(review_df['상품명'].unique())

20